# Step 3 Target analysis of 670 and 700nm excitation TA data of WL of SCy6803


### Package imports


In [ ]:
from glotaran.io import load_scheme
from pyglotaran_extras.compat import convert


def _case_study_convert(native_result, scheme):
    """Project native v0.8 results for legacy plotting while retaining native results."""
    import xarray as xr

    compat_result = convert(native_result)
    for dataset_label, dataset in compat_result.data.items():
        optimization_result = native_result.optimization_results[dataset_label]
        global_dimension = optimization_result.meta.global_dimension
        model_dimension = optimization_result.meta.model_dimension
        input_data = optimization_result.input_data
        residual = optimization_result.residuals
        if isinstance(input_data, xr.Dataset):
            input_data = input_data["data"]
        if isinstance(residual, xr.Dataset):
            residual = residual["residual"]
        fitted_data = input_data - residual
        if {"time", "spectral"}.issubset(fitted_data.dims):
            fitted_data = fitted_data.transpose("time", "spectral")
        dataset["fitted_data"] = fitted_data
        data_model = next(
            experiment.datasets[dataset_label]
            for experiment in scheme.experiments.values()
            if dataset_label in experiment.datasets
        )
        weight = xr.ones_like(residual)
        for weight_item in data_model.weights:
            selected = xr.ones_like(residual, dtype=bool)
            if weight_item.global_interval is not None:
                lower, upper = weight_item.global_interval
                selected = selected & (
                    (residual.coords[global_dimension] >= lower)
                    & (residual.coords[global_dimension] <= upper)
                )
            if weight_item.model_interval is not None:
                lower, upper = weight_item.model_interval
                selected = selected & (
                    (residual.coords[model_dimension] >= lower)
                    & (residual.coords[model_dimension] <= upper)
                )
            weight = weight * xr.where(selected, float(weight_item.value), 1.0)
        dataset["weight"] = weight
        dataset["weighted_residual"] = residual * weight
        dataset["clp"] = optimization_result.fit_decomposition.clp.rename(
            amplitude_label="clp_label"
        )
        dataset["matrix"] = optimization_result.fit_decomposition.matrix.rename(
            amplitude_label="clp_label"
        )
        kinetic_elements = [
            element
            for element in optimization_result.elements.values()
            if "compartment" in element.coords
        ]
        if kinetic_elements:
            dataset["species_concentration"] = xr.concat(
                [
                    element["concentrations"].rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            ).transpose(global_dimension, model_dimension, "species")
            dataset["species_associated_spectra"] = xr.concat(
                [element["amplitudes"].rename(compartment="species") for element in kinetic_elements],
                dim="species",
            ).transpose(global_dimension, "species")
            dataset["initial_concentration"] = xr.concat(
                [
                    element["initial_concentrations"]
                    .isel(activation=0, drop=True)
                    .rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
        spectral_elements = [
            element
            for element in optimization_result.elements.values()
            if "shape" in element.coords
        ]
        if spectral_elements:
            dataset["species_spectra"] = xr.concat(
                [
                    element["concentrations"].squeeze(drop=True).rename(shape="species")
                    for element in spectral_elements
                ],
                dim="species",
            ).transpose(model_dimension, "species")
        dataset.attrs["dataset_scale"] = optimization_result.meta.scale
    return compat_result


def _case_study_matrix_markdown(scheme, element_label, compartments=None):
    """Render a symbolic v0.8 kinetic rate map for legacy notebook display cells."""
    import pandas as pd

    element = scheme.library[element_label]
    compartments = list(compartments or element.compartments)
    table = [["" for _ in compartments] for _ in compartments]
    for (to_compartment, from_compartment), rate in element.rates.items():
        if to_compartment in compartments and from_compartment in compartments:
            table[compartments.index(to_compartment)][compartments.index(from_compartment)] = str(rate)
    return pd.DataFrame(table, index=compartments, columns=compartments).to_markdown()


from glotaran.io import load_parameters, save_result
import matplotlib.pyplot as plt
from pyglotaran_extras import plot_data_overview
from pyglotaran_extras import plot_fitted_traces, select_plot_wavelengths
from pyglotaran_extras.inspect import show_a_matrixes
from pyglotaran_extras.plotting.style import PlotStyle
from pyglotaran_extras.plotting.style import ColorCode
from cycler import cycler

### Data inspection


In [ ]:
DATA_PATH_670_1 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_reva.ascii'
DATA_PATH_670_2 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revb.ascii'
DATA_PATH_700_3 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revc.ascii'
DATA_PATH_700_4 = 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revd.ascii'

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH_700_4, nr_of_data_svd_vectors=4, linlog=False, linthresh=0.1, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True)

## Step 3a Target Analysis


### Model specification

This time the model and parameters are defined in each step, as the model and/or parameters may be tweaked in each step.


## Create scheme 1 with Bulk to RP2 and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step1_Bulk_RP2_CA_v08.yml')
target_scheme_parameters = load_parameters('models/target_step1_Bulk_RP2_CA.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result1_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result1 = _case_study_convert(target_result1_native, target_scheme)

In [ ]:
target_result1

### Plot result for interpretation


In [ ]:
custom_cycler1 = cycler(color=['g', 'b'])
custom_cycler2 = cycler(color=['g', 'b'], linestyle=['--'] * 2)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result1.data['700TR1'], target_result1.data['700TR2'], target_result1.data['670TR1'], target_result1.data['670TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler)

In [ ]:
target_result1.optimized_parameters

### Residual analysis of all data


In [ ]:
from custom_plotting import plot_svd_of_residual
(fig, axes) = plot_svd_of_residual([target_result1.data['670TR1'], target_result1.data['670TR2'], target_result1.data['700TR1'], target_result1.data['700TR2']], linlog=True, linthresh=10, index=0)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

## Step 3b Create scheme 2 introducing Red and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step2_Bulk_RP2_CA_Red_v08.yml')
target_scheme_parameters = load_parameters('models/target_step2_Bulk_RP2_CA_Red.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result2_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result2 = _case_study_convert(target_result2_native, target_scheme)

In [ ]:
target_result2

### Plot result for interpretation


In [ ]:
custom_cycler1 = cycler(color=['g', 'r', 'b'])
custom_cycler2 = cycler(color=['g', 'r', 'b'], linestyle=['--'] * 3)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result2.data['700TR1'], target_result2.data['700TR2'], target_result2.data['670TR1'], target_result2.data['670TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler)

In [ ]:
target_result2.optimized_parameters

### Residual analysis of all data


In [ ]:
(fig, axes) = plot_svd_of_residual([target_result2.data['670TR2'], target_result2.data['670TR1'], target_result2.data['700TR1'], target_result2.data['700TR2']], linlog=True, linthresh=10, index=0)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

## Step 3c Create scheme 3A introducing Ant1 and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step3A_Bulk_RP2_CA_Red_Ant1_v08.yml')
target_scheme_parameters = load_parameters('models/target_step3A_Bulk_RP2_CA_Red_Ant1.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result3_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result3 = _case_study_convert(target_result3_native, target_scheme)

In [ ]:
target_result3

### Plot result for interpretation


In [ ]:
myFRLcolors = [ColorCode.green, 'g', 'r', 'b']
custom_cycler1 = cycler(color=myFRLcolors)
custom_cycler2 = cycler(color=myFRLcolors, linestyle=['--'] * 4)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result3.data['670TR1'], target_result3.data['670TR2'], target_result3.data['700TR1'], target_result3.data['700TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler)

## Step 3d Create scheme 3B introducing RP1 and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step3B_Bulk_RP2_CA_Red_RP1_v08.yml')
target_scheme_parameters = load_parameters('models/target_step3B_Bulk_RP2_CA_Red_RP1.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result3_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result3 = _case_study_convert(target_result3_native, target_scheme)

In [ ]:
target_result3

### Plot result for interpretation


In [ ]:
myFRLcolors = ['g', 'r', ColorCode.cyan, 'b']
custom_cycler1 = cycler(color=myFRLcolors)
custom_cycler2 = cycler(color=myFRLcolors, linestyle=['--'] * 4)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result3.data['670TR1'], target_result3.data['670TR2'], target_result3.data['700TR1'], target_result3.data['700TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler)

## Step 3e Create scheme 3C introducing both Ant1 and RP1 and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step3C_Bulk_RP2_CA_Red_Ant1_RP1_v08.yml')
target_scheme_parameters = load_parameters('models/target_step3C_Bulk_RP2_CA_Red_Ant1_RP1.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4, 'RP1SADS': 'guide/global670and700_670TR2_clp_br4.ascii'}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result3_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result3 = _case_study_convert(target_result3_native, target_scheme)

In [ ]:
target_result3

### Plot result for interpretation


In [ ]:
myFRLcolors = [ColorCode.green, 'g', 'r', ColorCode.cyan, 'b']
custom_cycler1 = cycler(color=myFRLcolors)
custom_cycler2 = cycler(color=myFRLcolors, linestyle=['--'] * 5)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result3.data['670TR1'], target_result3.data['670TR2'], target_result3.data['700TR1'], target_result3.data['700TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler)

In [ ]:
target_result3.optimized_parameters

### Residual analysis of all data


In [ ]:
(fig, axes) = plot_svd_of_residual([target_result3.data['670TR2'], target_result3.data['670TR1'], target_result3.data['700TR1'], target_result3.data['700TR2']], linlog=True, linthresh=10, index=0)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

### Create the step 3C guidance data sets


In [ ]:
from glotaran.io import save_dataset
from glotaran.utils.io import create_clp_guide_dataset
for species in target_result3.data['670TR2'].species:
    clp_guide = create_clp_guide_dataset(target_result3.data['670TR2'], species.item())
    string_in_string = 'guide/target_step3C_Bulk_RP2_CA_Red_Ant1_RP1_clp_{}.ascii'.format(species.item())
    save_dataset(clp_guide.data, string_in_string, allow_overwrite=True)

## Step 3f Create scheme 4 introducing RC and free Chla and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_v08.yml')
target_scheme_parameters = load_parameters('models/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4, 'Ant1SADS': 'guide/target_step3C_Bulk_RP2_CA_Red_Ant1_RP1_clp_Ant1.ascii', 'RP1SADS': 'guide/target_step3C_Bulk_RP2_CA_Red_Ant1_RP1_clp_RP1.ascii', 'freeSADS': 'guide/global670and700_670TR2_clp_freeChla.ascii'}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result4_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=9, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result4 = _case_study_convert(target_result4_native, target_scheme)

In [ ]:
target_result4

### Plot result for interpretation


In [ ]:
myFRLcolors = [ColorCode.green, 'g', 'r', 'k', ColorCode.cyan, 'b', 'y']
custom_cycler1 = cycler(color=myFRLcolors)
custom_cycler2 = cycler(color=myFRLcolors, linestyle=['--'] * 7)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result4.data['670TR1'], target_result4.data['670TR2'], target_result4.data['700TR1'], target_result4.data['700TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler, show_DADS=False)

In [ ]:
target_result4.optimized_parameters

### Residual analysis of all data


In [ ]:
(fig, axes) = plot_svd_of_residual([target_result4.data['670TR1'], target_result4.data['670TR2'], target_result4.data['700TR1'], target_result4.data['700TR2']], linlog=True, linthresh=1, index=0)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

There is no more structure in the 1st LSV of the residual matrices. However, the RMSE of the 670TR1 is 10% larger than the RMSE of the other three data sets.


### Residual analysis of the 670 nm excitation TR1 data


In [ ]:
from custom_plotting import plot_residual_and_svd
(fig, axes) = plot_residual_and_svd([target_result4.data['670TR1']])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

LSV2 of the 670TR1 residual shows a trend on the timescale of 5 ps, and RSV2 is large around 700 nm. This can be solved by the splitting of the Red compartment in two parts. Therefore, we introduce a 2nd Red compartment. To smooth the SADS of the RC we will use a Spectral fit of the RC SADS estimated in step 4. To split the Red compartment we will guide the Red1 and Red2 SADS using the Spectral fit of the Red SADS from step 4 with two shifted guidance SADS.


### Create the step 4 SADS data sets


In [ ]:
for species in target_result4.data['670TR2'].species:
    clp_guide = create_clp_guide_dataset(target_result4.data['670TR2'], species.item())
    string_in_string = 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_{}.ascii'.format(species.item())
    save_dataset(clp_guide.data, string_in_string, allow_overwrite=True)

### Spectral fit of the estimated SADS of the RC with a sum of two skewed gaussians


In [ ]:
from glotaran.io import load_dataset, load_parameters
from pyglotaran_extras.plotting.style import PlotStyle
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)
dataset = load_dataset('guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC.ascii')
_ = plot_data_overview(dataset)

In [ ]:
spectral_model = load_scheme('models/spectral_model_v08.yml')
spectral_parameters = load_parameters('models/spectral_params_RC.yml')
spectral_scheme = spectral_model
spectral_scheme_parameters = spectral_parameters
spectral_scheme_datasets = {'dataset': dataset}
spectral_scheme_dry_run = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, maximum_number_function_evaluations=25, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=spectral_scheme load=PASS dry_run=PASS')
spectral_result_native = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, maximum_number_function_evaluations=25)
print('MIGRATION_VALIDATION scheme=spectral_scheme real_fit=PASS')
spectral_result = _case_study_convert(spectral_result_native, spectral_scheme)
spectral_result.optimized_parameters

In [ ]:
spectral_result

In [ ]:
from pyglotaran_extras.plotting.plot_overview import plot_overview
(fig, axes) = plot_overview(spectral_result.data['dataset'], linlog=False)

In [ ]:
spectral_result.optimized_parameters

In [ ]:
from glotaran.io import save_dataset
save_dataset(spectral_result.data['dataset'].fitted_data, 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC_fitted.ascii', allow_overwrite=True)

In [ ]:
spectral_resultRC = spectral_result

### Spectral fit of the Red SADS with two shifted species


In [ ]:
from glotaran.io import load_dataset, load_parameters
from pyglotaran_extras.plotting.style import PlotStyle
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)
dataset = load_dataset('guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_Red.ascii')
_ = plot_data_overview(dataset)

In [ ]:
spectral_model = load_scheme('models/spectral_model2splitRed_v08.yml')
spectral_parameters = load_parameters('models/spectral_parameters2splitRed.csv')
spectral_scheme = spectral_model
spectral_scheme_parameters = spectral_parameters
spectral_scheme_datasets = {'dataset': dataset}
spectral_scheme_dry_run = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=spectral_scheme load=PASS dry_run=PASS')
spectral_result_native = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets)
print('MIGRATION_VALIDATION scheme=spectral_scheme real_fit=PASS')
spectral_result = _case_study_convert(spectral_result_native, spectral_scheme)

In [ ]:
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)

### Spectral fits of the raw SADS of RC and Red


In [ ]:
(fig, axes) = plt.subplots(1, 2, figsize=(15, 4))
spectral_resultRC.data['dataset'].data.plot(ax=axes[0])
spectral_resultRC.data['dataset'].fitted_data.plot(ax=axes[0])
axes[0].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('SADS (mOD)')
axes[0].set_title('RC')
axes[0].axhline(0, color='k', linewidth=1)
spectral_result.data['dataset'].data.plot.line(x='spectral', ax=axes[1])
spectral_result.data['dataset'].fitted_data.plot.line(x='spectral', ax=axes[1])
spec1 = spectral_result.data['dataset'].clp[:, 0] * spectral_result.data['dataset'].species_spectra[:, 0] + spectral_result.data['dataset'].clp[:, 1] * spectral_result.data['dataset'].species_spectra[:, 1]
spec1.plot.line(x='spectral', ax=axes[1])
spec2 = spectral_result.data['dataset'].clp[:, 2] * spectral_result.data['dataset'].species_spectra[:, 2] + spectral_result.data['dataset'].clp[:, 3] * spectral_result.data['dataset'].species_spectra[:, 3]
spec2.plot.line(x='spectral', ax=axes[1])
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('SADS (mOD)')
axes[1].set_title('Red')
axes[1].axhline(0, color='k', linewidth=1)
axes[1].get_legend().remove()

In [ ]:
from glotaran.io import save_dataset
save_dataset(spec1, 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC_spec1_fitted.ascii', allow_overwrite=True)
save_dataset(spec2, 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC_spec2_fitted.ascii', allow_overwrite=True)

The guidance spectra created here are used in the next step to guide the Red1 and Red2 SADS.


## Step 3g.i Create scheme 5 introducing Red2 and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_v08.yml')
target_scheme_parameters = load_parameters('models/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4, 'RCSADS': 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC_fitted.ascii', 'Red1SADS': 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC_spec2_fitted.ascii', 'Red2SADS': 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_RC_spec1_fitted.ascii', 'Ant1SADS': 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_Ant1.ascii', 'freeSADS': 'guide/global670and700_670TR2_clp_freeChla.ascii'}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=7, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result5_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=7, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result5 = _case_study_convert(target_result5_native, target_scheme)

### Results and parameters


In [ ]:
target_result5

Note that this RMSE of is virtually the same as that reached in a global analysis with 5 and 4 lifetimes for the 670 and 700 nm excitation TA data, respectively.


### Residual analysis of the 670 nm excitation TR1 data


In [ ]:
(fig, axes) = plot_residual_and_svd([target_result5.data['670TR1']], indices=[0, 1])
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[2].annotate('C', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

The introduction of the 2nd Red compartment has led to a decrease in the RMSE and disappearance of the previous structure in LSV2 and RSV2 of 670TR1. Instead these are now dominated by the still imperfect description of the CA, cf. also the large residuals straddling time zero in (A) and (B).


### Plot result for interpretation


In [ ]:
myFRLcolors = [ColorCode.green, 'g', 'tab:orange', 'r', 'k', ColorCode.cyan, 'b', 'y']
custom_cycler1 = cycler(color=myFRLcolors)
custom_cycler2 = cycler(color=myFRLcolors, linestyle=['--'] * 8)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result4.data['670TR1'], target_result4.data['670TR2'], target_result4.data['700TR1'], target_result4.data['700TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler, show_DADS=False)

### Comparison of the estimated SADS (red) and the guidance spectra (black)

The guidance spectra are the SADS estimated with scheme 3C.
Alternatively, (smooth) shapes can be estimated with the help of spectral models or splines.


In [ ]:
(fig, axes) = plt.subplots(2, 3, figsize=(15, 7))
target_result5.data['RCSADS'].data.plot(ax=axes[0, 0])
target_result5.data['RCSADS'].fitted_data.plot(ax=axes[0, 0])
target_result5.data['Red1SADS'].data.plot(ax=axes[0, 1])
target_result5.data['Red1SADS'].fitted_data.plot(ax=axes[0, 1])
target_result5.data['Red2SADS'].data.plot(ax=axes[0, 2])
target_result5.data['Red2SADS'].fitted_data.plot(ax=axes[0, 2])
target_result5.data['Ant1SADS'].data.plot(ax=axes[1, 1])
target_result5.data['Ant1SADS'].fitted_data.plot(ax=axes[1, 1])
target_result5.data['freeSADS'].data.plot(ax=axes[1, 2])
target_result5.data['freeSADS'].fitted_data.plot(ax=axes[1, 2])
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('SADS (mOD)')
axes[0, 0].set_title('RC')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('SADS (mOD)')
axes[0, 1].set_title('Red1')
axes[0, 2].set_xlabel('')
axes[0, 2].set_ylabel('SADS (mOD)')
axes[0, 2].set_title('Red2')
axes[1, 0].set_xlabel('Wavelength (nm)')
axes[1, 0].set_ylabel('SADS (mOD)')
axes[1, 0].set_title('RP1')
axes[1, 1].set_xlabel('Wavelength (nm)')
axes[1, 1].set_ylabel('SADS (mOD)')
axes[1, 1].set_title('Ant1')
axes[1, 2].set_xlabel('Wavelength (nm)')
axes[1, 2].set_ylabel('SADS (mOD)')
axes[1, 2].set_title('free')

### Create the step 5 SADS data sets


In [ ]:
for species in target_result5.data['670TR2'].species:
    clp_guide = create_clp_guide_dataset(target_result5.data['670TR2'], species.item())
    string_in_string = 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_{}.ascii'.format(species.item())
    save_dataset(clp_guide.data, string_in_string, allow_overwrite=True)

Now in order to compute smooth guidance SADS we fit the estimated SADS of RC, Red1 and Red2 with a sum of two skewed gaussians, and employ these fits as the new guidance SADS, employing heavier weights of 0.4 and 0.7 for Red1 and Red2.


### Fit of the estimated SADS of RC, Red1, Red2, and free with a sum of two skewed gaussians


In [ ]:
from glotaran.io import load_dataset, load_parameters
from pyglotaran_extras.plotting.style import PlotStyle
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)
dataset = load_dataset('guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_RC.ascii')

In [ ]:
spectral_model = load_scheme('models/spectral_model_v08.yml')
spectral_parameters = load_parameters('models/spectral_params_RC.yml')
spectral_scheme = spectral_model
spectral_scheme_parameters = spectral_parameters
spectral_scheme_datasets = {'dataset': dataset}
spectral_scheme_dry_run = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, maximum_number_function_evaluations=25, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=spectral_scheme load=PASS dry_run=PASS')
spectral_result_native = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, maximum_number_function_evaluations=25)
print('MIGRATION_VALIDATION scheme=spectral_scheme real_fit=PASS')
spectral_result = _case_study_convert(spectral_result_native, spectral_scheme)
spectral_result.optimized_parameters

In [ ]:
spectral_result

In [ ]:
from pyglotaran_extras.plotting.plot_overview import plot_overview
(fig, axes) = plot_overview(spectral_result.data['dataset'], linlog=False)

In [ ]:
from glotaran.io import save_dataset
save_dataset(spectral_result.data['dataset'].fitted_data, 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_RC_fitted.ascii', allow_overwrite=True)

In [ ]:
spectral_resultRC = spectral_result

In [ ]:
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)
dataset = load_dataset('guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_Red1.ascii')

In [ ]:
spectral_model = load_scheme('models/spectral_model_v08.yml')
spectral_parameters = load_parameters('models/spectral_params.yml')
spectral_scheme = spectral_model
spectral_scheme_parameters = spectral_parameters
spectral_scheme_datasets = {'dataset': dataset}
spectral_scheme_dry_run = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=spectral_scheme load=PASS dry_run=PASS')
spectral_result_native = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets)
print('MIGRATION_VALIDATION scheme=spectral_scheme real_fit=PASS')
spectral_result = _case_study_convert(spectral_result_native, spectral_scheme)
spectral_result.optimized_parameters

In [ ]:
spectral_result

In [ ]:
from pyglotaran_extras.plotting.plot_overview import plot_overview
(fig, axes) = plot_overview(spectral_result.data['dataset'], linlog=False)

In [ ]:
from glotaran.io import save_dataset
save_dataset(spectral_result.data['dataset'].fitted_data, 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_Red1_fitted.ascii', allow_overwrite=True)

In [ ]:
spectral_resultRed1 = spectral_result

In [ ]:
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)
dataset = load_dataset('guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_Red2.ascii')

In [ ]:
spectral_model = load_scheme('models/spectral_model_v08.yml')
spectral_parameters = load_parameters('models/spectral_params.yml')
spectral_scheme = spectral_model
spectral_scheme_parameters = spectral_parameters
spectral_scheme_datasets = {'dataset': dataset}
spectral_scheme_dry_run = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=spectral_scheme load=PASS dry_run=PASS')
spectral_result_native = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets)
print('MIGRATION_VALIDATION scheme=spectral_scheme real_fit=PASS')
spectral_result = _case_study_convert(spectral_result_native, spectral_scheme)
spectral_result.optimized_parameters

In [ ]:
spectral_result

In [ ]:
from pyglotaran_extras.plotting.plot_overview import plot_overview
(fig, axes) = plot_overview(spectral_result.data['dataset'], linlog=False)

In [ ]:
from glotaran.io import save_dataset
save_dataset(spectral_result.data['dataset'].fitted_data, 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_Red2_fitted.ascii', allow_overwrite=True)

In [ ]:
spectral_resultRed2 = spectral_result

In [ ]:
plot_style = PlotStyle()
plt.rc('axes', prop_cycle=plot_style.cycler)
plt.rcParams['figure.figsize'] = (21, 14)
dataset = load_dataset('guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_free.ascii')

In [ ]:
spectral_model = load_scheme('models/spectral_model_v08.yml')
spectral_parameters = load_parameters('models/spectral_params_free.yml')
spectral_scheme = spectral_model
spectral_scheme_parameters = spectral_parameters
spectral_scheme_datasets = {'dataset': dataset}
spectral_scheme_dry_run = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, maximum_number_function_evaluations=25, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=spectral_scheme load=PASS dry_run=PASS')
spectral_result_native = spectral_scheme.optimize(parameters=spectral_scheme_parameters, datasets=spectral_scheme_datasets, maximum_number_function_evaluations=25)
print('MIGRATION_VALIDATION scheme=spectral_scheme real_fit=PASS')
spectral_result = _case_study_convert(spectral_result_native, spectral_scheme)
spectral_result.optimized_parameters

In [ ]:
spectral_result

In [ ]:
from pyglotaran_extras.plotting.plot_overview import plot_overview
(fig, axes) = plot_overview(spectral_result.data['dataset'], linlog=False)

In [ ]:
from glotaran.io import save_dataset
save_dataset(spectral_result.data['dataset'].fitted_data, 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_free_fitted.ascii', allow_overwrite=True)

### Spectral fits of the step5 SADS of RC, Red1, Red2


In [ ]:
(fig, axes) = plt.subplots(1, 3, figsize=(15, 4))
spectral_resultRC.data['dataset'].data.plot(ax=axes[0])
spectral_resultRC.data['dataset'].fitted_data.plot(ax=axes[0])
spectral_resultRed1.data['dataset'].data.plot(ax=axes[1])
spectral_resultRed1.data['dataset'].fitted_data.plot(ax=axes[1])
spectral_resultRed2.data['dataset'].data.plot(ax=axes[2])
spectral_resultRed2.data['dataset'].fitted_data.plot(ax=axes[2])
axes[0].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('SADS (mOD)')
axes[0].set_title('RC')
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('SADS (mOD)')
axes[1].set_title('Red1')
axes[2].set_xlabel('Wavelength (nm)')
axes[2].set_ylabel('SADS (mOD)')
axes[2].set_title('Red2')

## Step 3g.ii Use scheme 5 with smooth guidance SADS and optimize it


In [ ]:
target_scheme = load_scheme('models/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_v08.yml')
target_scheme_parameters = load_parameters('models/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2.csv')
target_scheme_datasets = {'670TR1': DATA_PATH_670_1, '670TR2': DATA_PATH_670_2, '700TR1': DATA_PATH_700_3, '700TR2': DATA_PATH_700_4, 'RCSADS': 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_RC_fitted.ascii', 'Red1SADS': 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_Red1_fitted.ascii', 'Red2SADS': 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_Red2_fitted.ascii', 'Ant1SADS': 'guide/target_step4_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_clp_Ant1.ascii', 'freeSADS': 'guide/target_step5_Bulk_RP2_CA_Red_Ant1_RP1_RC_free_Red2_clp_free_fitted.ascii'}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=7, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=7, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result = _case_study_convert(target_result_native, target_scheme)

### Results and parameters


In [ ]:
target_result

To save the results of the optimization we can use the `save_result` command.

Because it saves _everything_ it consumes about 52MB of disk space per save.


In [ ]:
save_result(result=target_result_native, result_path='results/target670and700/result.yaml', allow_overwrite=True)

### Plot result for interpretation


In [ ]:
myFRLcolors = [ColorCode.green, 'g', 'tab:orange', 'r', 'k', ColorCode.cyan, 'b', 'y']
custom_cycler1 = cycler(color=myFRLcolors)
custom_cycler2 = cycler(color=myFRLcolors, linestyle=['--'] * 8)
custom_cyclers = (custom_cycler1, custom_cycler1, custom_cycler2, custom_cycler2)
from custom_plotting import plot_concentration_and_spectra
_ = plot_concentration_and_spectra([target_result3.data['670TR1'], target_result3.data['670TR2'], target_result3.data['700TR1'], target_result3.data['700TR2']], cycler=custom_cyclers, das_cycler=PlotStyle().cycler)

### Residual analysis of all data


In [ ]:
(fig, axes) = plot_svd_of_residual([target_result.data['670TR1'], target_result.data['670TR2'], target_result.data['700TR1'], target_result.data['700TR2']], linlog=True, linthresh=1, index=0)
axes[0].annotate('A', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)
axes[1].annotate('B', xy=(-0.1, 1), xycoords='axes fraction', fontsize=16)

### Comparison of the estimated SADS (red) and the guidance spectra (black)

The guidance spectra are the SADS estimated with scheme 3C.
Alternatively, (smooth) shapes can be estimated with the help of spectral models or splines.


In [ ]:
(fig, axes) = plt.subplots(2, 3, figsize=(15, 7))
target_result.data['RCSADS'].data.plot(ax=axes[0, 0])
target_result.data['RCSADS'].fitted_data.plot(ax=axes[0, 0])
target_result.data['Red1SADS'].data.plot(ax=axes[0, 1])
target_result.data['Red1SADS'].fitted_data.plot(ax=axes[0, 1])
target_result.data['Red2SADS'].data.plot(ax=axes[0, 2])
target_result.data['Red2SADS'].fitted_data.plot(ax=axes[0, 2])
target_result.data['Ant1SADS'].data.plot(ax=axes[1, 1])
target_result.data['Ant1SADS'].fitted_data.plot(ax=axes[1, 1])
target_result.data['freeSADS'].data.plot(ax=axes[1, 2])
target_result.data['freeSADS'].fitted_data.plot(ax=axes[1, 2])
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('SADS (mOD)')
axes[0, 0].set_title('RC')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('SADS (mOD)')
axes[0, 1].set_title('Red1')
axes[0, 2].set_xlabel('')
axes[0, 2].set_ylabel('SADS (mOD)')
axes[0, 2].set_title('Red2')
axes[1, 0].set_xlabel('Wavelength (nm)')
axes[1, 0].set_ylabel('SADS (mOD)')
axes[1, 0].set_title('RP1')
axes[1, 1].set_xlabel('Wavelength (nm)')
axes[1, 1].set_ylabel('SADS (mOD)')
axes[1, 1].set_title('Ant1')
axes[1, 2].set_xlabel('Wavelength (nm)')
axes[1, 2].set_ylabel('SADS (mOD)')
axes[1, 2].set_title('free')

In [ ]:
(fig, axes) = plt.subplots(1, 3, figsize=(15, 4))
target_result.data['RCSADS'].data.plot(ax=axes[0])
target_result.data['RCSADS'].fitted_data.plot(ax=axes[0])
target_result.data['Red1SADS'].data.plot(ax=axes[1])
target_result.data['Red1SADS'].fitted_data.plot(ax=axes[1])
target_result.data['Red2SADS'].data.plot(ax=axes[2])
target_result.data['Red2SADS'].fitted_data.plot(ax=axes[2])
axes[0].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('SADS (mOD)')
axes[0].set_title('RC')
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('SADS (mOD)')
axes[1].set_title('Red1')
axes[2].set_xlabel('Wavelength (nm)')
axes[2].set_ylabel('SADS (mOD)')
axes[2].set_title('Red2')
axes[0].axhline(0, color='k', linewidth=1)
axes[1].axhline(0, color='k', linewidth=1)
axes[2].axhline(0, color='k', linewidth=1)

In [ ]:
target_result.optimized_parameters

### Result plots


<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>


### Fit quality


In [ ]:
target_result_TA = (target_result.data['670TR1'], target_result.data['670TR2'], target_result.data['700TR1'], target_result.data['700TR2'])
wavelengths = select_plot_wavelengths(target_result_TA, equidistant_wavelengths=True)
plot_fitted_traces(target_result_TA, wavelengths, linlog=True, linthresh=1)

The above command `plot_fitted_traces` is used to plot a selection of traces for a set of wavelengths (autogenerated using the `select_plot_wavelengths` function).
To show to make a manual selection of traces, and 'dress up the plot' see the code below, which reproduces Figure 2 of the iScience paper.


In [ ]:
from pyglotaran_extras.plotting.style import ColorCode as cc
from custom_plotting import plot_fitted_traces_iscience
(fig, ax_) = plot_fitted_traces_iscience(target_result_TA, [685, 700, 720, 760], linlog=True, linthresh=1, axes_shape=(2, 2), figsize=(6, 4), title='', per_axis_legend=True, cycler=cycler(color=[cc.grey, cc.black, cc.grey, cc.black, cc.orange, cc.red, cc.orange, cc.red]))

### Overview 670 exc


In [ ]:
overview_670TR1 = plot_overview(target_result.data['670TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=[ColorCode.green, 'g', ColorCode.orange, 'r', 'k', ColorCode.cyan, 'b', 'y']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

In [ ]:
overview_670TR2 = plot_overview(target_result.data['670TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=[ColorCode.green, 'g', ColorCode.orange, 'r', 'k', ColorCode.cyan, 'b', 'y']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

### Overview 700 exc


In [ ]:
overview_700TR1 = plot_overview(target_result.data['700TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=[ColorCode.green, 'g', ColorCode.orange, 'r', 'k', ColorCode.cyan, 'b', 'y']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

In [ ]:
overview_700TR2 = plot_overview(target_result.data['700TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=2, linlog=False, linthresh=1, cycler=cycler(color=[ColorCode.green, 'g', ColorCode.orange, 'r', 'k', ColorCode.cyan, 'b', 'y']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

### Amplitude matrices


In [ ]:
show_a_matrixes(target_result)

### K-matrix of the target model


In [ ]:
_case_study_matrix_markdown(target_scheme, 'complex670', ['Ant1', 'Bulk', 'Red1', 'Red2', 'RC', 'RP1', 'RP2', 'free'])